In [1]:
import os
import glob
import datetime

import numpy as np
import pandas as pd

import jax
import numpyro

import hssm
import arviz as az
from scipy.stats import gaussian_kde

import matplotlib.pyplot as plt
import seaborn as sns

import sqlite3

/Users/javierrojas/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# general functions

In [2]:
def get_fitted_parameters(df, participant_id):
    subset = df[df['participant_id'] == participant_id]
    v = subset[subset['param'] == 'v']['mean'].values[0]
    a = subset[subset['param'] == 'a']['mean'].values[0]
    z = subset[subset['param'] == 'z']['mean'].values[0]
    t = subset[subset['param'] == 't']['mean'].values[0]
    return v, a, z, t

# define fitters

In [3]:
def fit_hssm_participant(df, participant_column):
    all_summaries = []
    all_inferences = {}

    for nsub, isub in enumerate(df[participant_column].unique()):
        print(f"___Participant {isub}, {nsub+1}/{df[participant_column].nunique()}___")

        df_sub = df[df[participant_column] == isub].drop(columns=[participant_column])

        print("Median RT =", np.median(df_sub['rt']))
        print("N trials =", len(df_sub))

        model = hssm.HSSM(
            model="ddm",
            data=df_sub,
        )

        infer_data_sub = model.sample(
            cores=3,
            chains=3,
            draws=300,
            tune=1000,
            idata_kwargs=dict(log_likelihood=True),
            progressbar=True,
            target_accept=0.99,
        )

        all_inferences[isub] = infer_data_sub

        summary_df = (
            az.summary(infer_data_sub)
              .reset_index()
              .rename(columns={'index': 'param'})
        )
        summary_df['participant_id'] = isub
        all_summaries.append(summary_df)

    all_summaries_df = pd.concat(all_summaries, ignore_index=True)
    return all_summaries_df, all_inferences

In [4]:
def fit_hssm_participant_th(df, participant_column):
    all_summaries = []
    all_inferences = {}   # <- store InferData here

    for nsub, isub in enumerate(df[participant_column].unique()):
        print(f"___Participant {isub}, {nsub+1}/{df[participant_column].nunique()}___")

        df_sub = df[df[participant_column] == isub].drop(columns=[participant_column])

        print("Median RT =", np.median(df_sub['rt']))
        print("N trials =", len(df_sub))

        model = hssm.HSSM(
            data=df_sub,
            model="ddm",
            include=[
                {
                    "name": "a",
                    "formula": "a ~ 1 + ab_nominal",
                    "prior": {
                        # All ways to specify priors in the non-regression case work the same way here.
                        "Intercept": {"name": "Normal", "mu": 1.5, "sigma": 1.0},
                        "ab_nominal": {"name": "Normal", "mu": 0, "sigma": 1.0},
                    },
                    "link": "identity",
                }
            ],
        )

        infer_data_sub = model.sample(
            cores=3,
            chains=3,
            draws=300,
            tune=1000,
            idata_kwargs=dict(log_likelihood=True),
            progressbar=True,
            target_accept=0.99,
        )

        all_inferences[isub] = infer_data_sub

        summary_df = (
            az.summary(infer_data_sub)
              .reset_index()
              .rename(columns={'index': 'param'})
        )
        summary_df['participant_id'] = isub
        all_summaries.append(summary_df)

    all_summaries_df = pd.concat(all_summaries, ignore_index=True)
    return all_summaries_df, all_inferences

In [5]:
def fit_hssm_participant_th(df, participant_column):
    all_summaries = []
    all_inferences = {}   # <- store InferData here

    for nsub, isub in enumerate(df[participant_column].unique()):
        print(f"___Participant {isub}, {nsub+1}/{df[participant_column].nunique()}___")

        df_sub = df[df[participant_column] == isub].drop(columns=[participant_column])

        print("Median RT =", np.median(df_sub['rt']))
        print("N trials =", len(df_sub))

        model = hssm.HSSM(
            data=df_sub,
            model="ddm",
            include=[
                {
                    "name": "a",
                    "formula": "a ~ 0 + I(1 + ab_nominal)",
                    "prior": {
                        "I(1 + ab_nominal)": {"name": "Normal", "mu": 1.5, "sigma": 1.0},
                    },
                    "link": "identity",
                }
            ],
        )

        infer_data_sub = model.sample(
            cores=3,
            chains=3,
            draws=300,
            tune=1000,
            idata_kwargs=dict(log_likelihood=True),
            progressbar=True,
            target_accept=0.99,
        )

        all_inferences[isub] = infer_data_sub

        summary_df = (
            az.summary(infer_data_sub)
              .reset_index()
              .rename(columns={'index': 'param'})
        )
        summary_df['participant_id'] = isub
        all_summaries.append(summary_df)

    all_summaries_df = pd.concat(all_summaries, ignore_index=True)
    return all_summaries_df, all_inferences

In [6]:
def fit_hssm_participant_v(df, participant_column):
    all_summaries = []
    all_inferences = {}   # <- store InferData here

    for nsub, isub in enumerate(df[participant_column].unique()):
        print(f"___Participant {isub}, {nsub+1}/{df[participant_column].nunique()}___")

        df_sub = df[df[participant_column] == isub].drop(columns=[participant_column])

        print("Median RT =", np.median(df_sub['rt']))
        print("N trials =", len(df_sub))

        model = hssm.HSSM(
            data=df_sub,
            model="ddm",
            include=[
                {
                    "name": "v",
                    "formula": "v ~ 1 + ab_nominal",
                    "prior": {
                        # All ways to specify priors in the non-regression case work the same way here.
                        "Intercept": {"name": "Normal", "mu": 1.5, "sigma": 1.0},
                        "ab_nominal": {"name": "Normal", "mu": 0, "sigma": 1.0},
                    },
                    "link": "identity",
                }
            ],
        )

        infer_data_sub = model.sample(
            cores=3,
            chains=3,
            draws=300,
            tune=1000,
            idata_kwargs=dict(log_likelihood=True),
            progressbar=True,
            target_accept=0.99,
        )

        all_inferences[isub] = infer_data_sub

        summary_df = (
            az.summary(infer_data_sub)
              .reset_index()
              .rename(columns={'index': 'param'})
        )
        summary_df['participant_id'] = isub
        all_summaries.append(summary_df)

    all_summaries_df = pd.concat(all_summaries, ignore_index=True)
    return all_summaries_df, all_inferences

# define simulators

In [7]:
def simulate_participant_ddm(participant_id, df, size=300):
    v, a, z, t = get_fitted_parameters(df, participant_id)
    v = np.repeat(v, size)          # drift rate
    a = a                           # boundary
    z = z                           # starting point
    t = t                           # non-decision time
    true_values = np.column_stack([v, np.repeat([[a, z, t]], size, axis=0)])

    dataset = hssm.simulate_data(
        model="ddm",
        theta=true_values,
        size=1,
    )

    dataset["participant_id"] = str(participant_id)
    return dataset

In [8]:
def simulate_participant_ddm_th(participant_id, df, size=300):
    v, a0, z, t = get_fitted_parameters(df, participant_id)

    df_sub = df[df["participant_id"] == participant_id].iloc[:size]
    ab = df_sub["ab_nominal"].values

    beta_a = df_sub["beta_a"] if "beta_a" in df_sub else 0.0
    a = a0 + beta_a * ab
    v = np.repeat(v, size)

    theta = np.column_stack([v, a, np.repeat(z, size), np.repeat(t, size)])

    dataset = hssm.simulate_data(
        model="ddm",
        theta=theta,
        size=1,
    )

    dataset["participant_id"] = str(participant_id)
    dataset["ab_nominal"] = ab
    return dataset

In [9]:
def simulate_participant_ddm_v(participant_id, df, size=300):
    v0, a, z, t = get_fitted_parameters(df, participant_id)

    df_sub = df[df["participant_id"] == participant_id].iloc[:size]
    ab = df_sub["ab_nominal"].values

    beta_v = df_sub["beta_v"] if "beta_v" in df_sub else 0.0
    v = v0 + beta_v * ab
    a = np.repeat(a, size)

    theta = np.column_stack([v, a, np.repeat(z, size), np.repeat(t, size)])

    dataset = hssm.simulate_data(
        model="ddm",
        theta=theta,
        size=1,
    )

    dataset["participant_id"] = str(participant_id)
    dataset["ab_nominal"] = ab
    return dataset

# test

In [10]:
df_exp = pd.read_csv('hssm_exp4_data.csv')
df_sim = pd.read_csv('hssm_simulated_data.csv')

In [11]:
query = "SELECT * FROM summaries"
conn = sqlite3.connect("hssm_fits.db")
df_results = pd.read_sql_query(query, conn)
conn.close()

In [12]:
# 1. Pick a participant to "clone"
target_sub = df_exp['participant_id'].unique()[0]

# 2. Simulate new data using your function
# This uses the means from your empirical fit as the 'Ground Truth'
print(f"Simulating recovery data for Participant {target_sub}...")
df_simulated = simulate_participant_ddm_v(target_sub, df_results, size=400)

# 3. Fit the simulated data using your fitter
# This will try to recover the parameters we just used to build the data
print("Fitting simulated data...")
recovered_summary, recovered_inferences = fit_hssm_participant_v(
    df_simulated, 
    participant_column='participant_id'
)

# 4. Compare True vs. Recovered
v_true, a_true, z_true, t_true = get_fitted_parameters(df_results, target_sub)

print("\n--- RECOVERY RESULTS ---")
results_comp = pd.DataFrame({
    'Parameter': ['v_Intercept', 'a', 't'],
    'True_Value': [v_true, a_true, t_true],
    'Recovered_Mean': [
        recovered_summary.loc[recovered_summary['param'] == 'v_Intercept', 'mean'].values[0],
        recovered_summary.loc[recovered_summary['param'] == 'a', 'mean'].values[0],
        recovered_summary.loc[recovered_summary['param'] == 't', 'mean'].values[0]
    ]
})
print(results_comp)

# 5. Visual Check
az.plot_posterior(recovered_inferences[target_sub], var_names=['v_Intercept'], ref_val=v_true)
plt.title(f"Recovery of v_Intercept (True: {v_true:.2f})")
plt.show()

Simulating recovery data for Participant 708...


KeyError: 'ab_nominal'